# Chirurgie à chaud EN COURS d'un vrai entraînement + analyse causale de la couche greffée

Fusionne trois expériences du plan de session (P1+P2+P3) en une seule, cohérente :

1. **Phase A** : entraîner le char-LM (TinyShakespeare, dim=256/4 têtes/4 couches,
   `batched_attn=true`) pendant 10 000 pas -- la moitié du run de référence de
   `real_llm.ipynb` (déjà validé : val loss finale 1.6229 à 20 000 pas, Δ0.0033
   nats avec le miroir PyTorch).
2. **Baseline** : recherche de circuit d'induction sur texte réel (noms de
   personnages répétés dans TinyShakespeare, ex. "MENENIUS:") sur le modèle à
   4 couches, à mi-parcours -- `greedy_patch_search!`/`backward_prune!` avec le
   nouveau kwarg `metric` (recovery restreinte à une ligne, comme `induction.ipynb`
   mais sur du texte réel plutôt que sur une tâche synthétique).
3. **Chirurgie** : `insert_block!` (avec le nouveau kwarg `batched_attn=true` pour
   rester homogène) insère une 5ème couche EN COURS d'entraînement -- preuve F1
   (texte généré identique bit-à-bit juste avant/après insertion).
4. **Phase B** : poursuite de l'entraînement (5 couches) pour les 10 000 pas
   restants, moments AdamW fusionnés par nom de paramètre (patron F4 de
   `test/test_surgery.jl`), pas de réinitialisation du pas `t` global.
5. **Final** : même recherche de circuit sur les mêmes fenêtres gelées, maintenant
   sur 5 couches -- la nouvelle couche a-t-elle acquis une responsabilité causale
   mesurable ? Verdict honnête contre des critères falsifiables, quel que soit
   le résultat.

Aucune modification de `real_llm.ipynb` (artefact de parité déjà validé, laissé
intact). Deux correctifs `src/` de cette session sont des prérequis directs :
`greedy_patch_search!`/`backward_prune!` ne perdent plus le patch d'un site déjà
retenu quand un candidat amont est testé (bug trouvé et corrigé le 2026-07-10),
et `insert_block!`/les deux fonctions de recherche acceptent maintenant
`batched_attn`/`metric`.

In [1]:
using NeuroDSL, Random, Statistics, Printf, StatsPlots

dev = NeuroDSL.Backend.CUDADevice()
ns = :real_llm_surgery
println("Device: ", dev)

Device: NeuroDSL.Backend.CUDADevice()


## 1. Corpus : TinyShakespeare (téléchargé une seule fois, hors ligne ensuite)

In [2]:
using Downloads

const CORPUS_URL  = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
const CORPUS_PATH = joinpath(@__DIR__, "data", "tinyshakespeare", "input.txt")

function load_corpus(path::String, url::String)
    if !isfile(path)
        mkpath(dirname(path))
        Downloads.download(url, path)
    end
    text = read(path, String)
    println("Corpus chargé : ", length(text), " caractères depuis ", path)
    return text
end

text = load_corpus(CORPUS_PATH, CORPUS_URL)
println(first(text, 200))

Corpus chargé : 1115394 caractères depuis C:\Users\Nevermind\Desktop\NeuroDSL\notebook\data\tinyshakespeare\input.txt
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


## 2. Tokenizer caractère (vrai texte -> vrais IDs, pas de BPE, zéro dépendance nouvelle)

In [3]:
function build_char_tokenizer(text::String)
    chars = sort(unique(collect(text)))
    stoi = Dict(c => i for (i, c) in enumerate(chars))
    return chars, stoi
end

encode(text::AbstractString, stoi::Dict{Char,Int}) = [stoi[c] for c in text]
decode(ids::AbstractVector{<:Integer}, chars::Vector{Char}) = String(chars[ids])

chars, stoi = build_char_tokenizer(text)
vocab_size = length(chars)
println("vocab_size = ", vocab_size)
println("10 premiers caractères (triés) = ", chars[1:10])
println("(à comparer avec les 10 premiers caractères imprimés par real_llm_py.py -- doivent être identiques)")

vocab_size = 65
10 premiers caractères (triés) = ['\n', ' ', '!', '$', '&', '\'', ',', '-', '.', '3']
(à comparer avec les 10 premiers caractères imprimés par real_llm_py.py -- doivent être identiques)


## 3. Split train/validation (90/10 par position -- le val est la FIN du corpus, jamais vue à l'entraînement)

In [4]:
data = encode(text, stoi)
n_total = length(data)
n_train = floor(Int, 0.9 * n_total)
train_ids = data[1:n_train]
val_ids   = data[n_train+1:end]
println("train: ", length(train_ids), " caractères  |  val: ", length(val_ids), " caractères")

train: 1003854 caractères  |  val: 111540 caractères


## 4. Construction du graphe (copie directe de `build_induction_graph`, généralisée à un vrai vocabulaire/contexte)

In [5]:
function build_char_lm_graph(dev, ns::Symbol; vocab_size::Int, dim::Int, n_heads::Int,
                              hidden_dim::Int, n_layers::Int, block_size::Int, batched::Bool=true)
    g = NeuroDSL.NeuroGraph(namespace=ns, device=dev)
    NeuroDSL.set!(g, :token_ids, ones(Int, block_size); atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
    tok_emb = NeuroDSL.Embedding(vocab_size, dim)(g, :token_ids, :tok; namespace=ns)
    pos_emb = NeuroDSL.Embedding(block_size, dim)(g, :pos_ids, :pos; namespace=ns)
    x = :embed_sum
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(x, [tok_emb, pos_emb], :add; namespace=ns))
    # `batched` (défaut true) exposé en kwarg pour P1-bis (graphe jumeau non-batché,
    # comparaison de coût/cône de patch) -- src/layers.jl, conçu avec Fable.
    out = NeuroDSL.LlamaModel(n_layers, dim, n_heads, hidden_dim; batched_attn=batched)(g, x; namespace=ns)
    logits = NeuroDSL.Linear(dim, vocab_size)(g, out, :lm_head; namespace=ns)
    NeuroDSL.set!(g, :labels, ones(Int, block_size); atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [logits, :labels], :cross_entropy; namespace=ns))
    return g, logits
end

# ── Hyperparamètres (voir plan : dimensionnés pour rester bien sous le max GPU
# déjà confirmé cette session -- dim=1024 en forward+backward réel) ──────────
block_size = 256
dim        = 256
n_heads    = 4
hidden_dim = 512
n_layers   = 4

g, logits_sym = build_char_lm_graph(dev, ns; vocab_size=vocab_size, dim=dim, n_heads=n_heads,
                                     hidden_dim=hidden_dim, n_layers=n_layers, block_size=block_size)
ps = NeuroDSL.params(g; namespace=ns)
n_scalars = sum(length(p.value) for p in ps)
println("Graphe : ", length(g.nodes[ns]), " nœuds, ", length(ps), " tenseurs de paramètres, ",
        n_scalars, " scalaires (~", round(n_scalars/1e6, digits=2), "M)")

Graphe : 220 nœuds, 40 tenseurs de paramètres, 2722369 scalaires (~2.72M)


## 5. Échantillonnage de fenêtres réelles + perte de validation + génération autorégressive

In [6]:
function sample_window(rng, ids::Vector{Int}, block_size::Int)
    i = rand(rng, 1:(length(ids) - block_size))
    tokens = ids[i:i+block_size-1]
    labels = ids[i+1:i+block_size]
    return tokens, labels
end

# 64 fenêtres FIXES également espacées dans le split val -- déterministe,
# comparable entre checkpoints. Jamais de backward_graph! ici (pas de fuite
# du val dans les gradients) -- donc `demand_release!` (src/demand_release.jl)
# est sûr : libère les activations intermédiaires au fil du calcul au lieu de
# les garder résidentes jusqu'au prochain train_char_lm! (mêmes résultats,
# vérifié bit-à-bit cette session -- réduit juste le pic VRAM de cet appel).
function val_loss(g::NeuroDSL.NeuroGraph, ns::Symbol; val_ids::Vector{Int}, block_size::Int, n_windows::Int=64)
    max_start = length(val_ids) - block_size
    starts = round.(Int, range(1, max_start, length=n_windows))
    total = 0.0
    for i in starts
        tokens = val_ids[i:i+block_size-1]
        labels = val_ids[i+1:i+block_size]
        NeuroDSL.set!(g, :token_ids, tokens; atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.set!(g, :labels, labels; atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.invalidate_all!(g; namespace=ns)
        loss_val = NeuroDSL.demand_release!(g, :loss; namespace=ns)
        total += Float64(sum(Array(loss_val)))
    end
    return total / n_windows
end

# Génération autorégressive -- échantillonnage AVEC TEMPÉRATURE (pas argmax) :
# l'argmax sur un char-LM dégénère quasi systématiquement en boucles
# répétitives ("the the the..."), donnant une fausse impression d'échec alors
# que la distribution apprise est bonne. C'est ce que nanoGPT/char-rnn font
# pour leurs démos.
function generate_text(g::NeuroDSL.NeuroGraph, logits_sym::Symbol, ns::Symbol,
                        stoi::Dict{Char,Int}, chars::Vector{Char};
                        seed_text::String="\n", n_chars::Int=300, temperature::Float32=0.8f0,
                        block_size::Int, rng=MersenneTwister(777), use_argmax::Bool=false)
    ctx = encode(seed_text, stoi)
    generated = Char[]
    for _ in 1:n_chars
        window = length(ctx) > block_size ? ctx[end-block_size+1:end] : ctx
        t = length(window)
        NeuroDSL.set!(g, :token_ids, window; atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.set!(g, :pos_ids, collect(1:t); atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.invalidate_all!(g; namespace=ns)
        row = Array(NeuroDSL.demand_release!(g, logits_sym; namespace=ns))[end, :]
        local next_id
        if use_argmax
            next_id = argmax(row)
        else
            p = exp.((row .- maximum(row)) ./ temperature)
            p ./= sum(p)
            r = rand(rng)
            cum = 0.0f0
            next_id = length(p)
            for (idx, pi) in enumerate(p)
                cum += pi
                if r <= cum
                    next_id = idx
                    break
                end
            end
        end
        push!(ctx, next_id)
        push!(generated, chars[next_id])
    end
    return String(generated)
end

generate_text (generic function with 1 method)

## 6. Vérification de sanité : perte initiale ≈ ln(vocab_size)

In [7]:
rng_check = MersenneTwister(1)
tokens0, labels0 = sample_window(rng_check, train_ids, block_size)
NeuroDSL.set!(g, :token_ids, tokens0; atom_type=NeuroDSL.Datom, namespace=ns)
NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
NeuroDSL.set!(g, :labels, labels0; atom_type=NeuroDSL.Datom, namespace=ns)
NeuroDSL.invalidate_all!(g; namespace=ns)
loss0 = Float64(sum(Array(NeuroDSL.demand!(g, :loss; namespace=ns))))
@printf("Perte initiale mesurée : %.4f   (attendu ln(%d) = %.4f)\n", loss0, vocab_size, log(vocab_size))
@assert abs(loss0 - log(vocab_size)) < 0.5 "Perte initiale trop loin de ln(vocab_size) -- vérifier le câblage avant d'entraîner"

println("\n--- Échantillon AVANT tout entraînement (poids aléatoires) ---")
sample_before = generate_text(g, logits_sym, ns, stoi, chars; block_size=block_size, rng=MersenneTwister(777))
println(sample_before)

Perte initiale mesurée : 4.2947   (attendu ln(65) = 4.1744)

--- Échantillon AVANT tout entraînement (poids aléatoires) ---
,3sqoj
FVJdi$AVxeOevAP!K,yqFCH!vHF!yc&!HqFE!Y$!&
qUdtj.glnYWAA!uhY-xxHWiOeXxJl;Z.mrV
YoNSK
xZf3X
mpv:KTcjvkQo
&xy
&nMMCFisH3rK3XJm'YS-kxhhRXL3awBTYANmqbhsLsZTdyh
h&V!Fe
 GgXS
;i'MHqs$vdH3lO,yKWrAUtorFBW3P,vHprDYJbU
vZx-!HcxvCEKlTvWp?vv-;vmxQk.zQ!ukj$-c&zH AKzYDYIHKXBNjlS3ojk jQadS3WD!k&kf&qI.IZgqy!v


## 7. Budget de calcul (estimé par Fable, à partir du chronométrage réel de `real_llm.ipynb`)

Référence mesurée : 505.7 s / 20 000 pas = 25.3 ms/pas à 4 couches (GPU RTX A5500).
Estimation pour cette expérience : Phase A (10k pas, 4 couches) ≈ 253 s ; Phase B
(10k pas, 5 couches) ≈ 316 s (borne sup à 31.6 ms/pas) ; les deux recherches de
circuit (gloutonne + élagage, 3 fenêtres, 16 puis 20 candidats) ≈ moins d'une
minute au total (chaque mesure est un `demand!` incrémental + une restauration
par copie de cache, pas un forward complet). **Total estimé : ~12-16 minutes.**
Le coût est dominé par l'entraînement, pas par l'interprétabilité -- c'est un
résultat en soi, cohérent avec la thèse du framework.

## 8. Phase A : 10 000 premiers pas (4 couches)

`train_char_lm!` v2 : accepte maintenant `rng` (objet, pas une graine entière),
`t0` (pas de départ, pour que le compteur AdamW `t` ne se réinitialise jamais à
la frontière de la greffe) et `moments` (`Dict{Symbol,Tuple}` keyé par NOM de
paramètre, patron F4 de `test/test_surgery.jl` -- après `insert_block!`,
`params(g)` peut réordonner, donc réutiliser des `Vector`s positionnels d'une
phase à l'autre serait incorrect). Retourne `moments` et `rng` en plus des
métriques habituelles, pour les transmettre tels quels à la Phase B.

In [8]:
function train_char_lm!(g::NeuroDSL.NeuroGraph, ns::Symbol, logits_sym::Symbol,
                         stoi::Dict{Char,Int}, chars::Vector{Char};
                         train_ids::Vector{Int}, val_ids::Vector{Int}, block_size::Int,
                         n_steps::Int, lr::Float32=1f-3, b1::Float32=0.9f0, b2::Float32=0.999f0,
                         eps_v::Float32=1f-8, clip::Float32=1f0, wd::Float32=0f0,
                         rng::MersenneTwister=MersenneTwister(123), t0::Int=0,
                         moments::Union{Nothing,Dict{Symbol,Tuple{Any,Any}}}=nothing,
                         val_every::Int=500, sample_steps=(), norm_watch::Vector{Symbol}=Symbol[])
    dev = g.device
    ps = NeuroDSL.params(g; namespace=ns)
    # Moments keyés par NOM (pas par position) -- patron F4 (test/test_surgery.jl) :
    # un paramètre déjà connu (avant une éventuelle greffe) reprend ses moments
    # exacts ; un paramètre nouveau (apporté par insert_block!) démarre à zéro.
    m1s = Vector{Any}(undef, length(ps))
    m2s = Vector{Any}(undef, length(ps))
    for (i, p) in enumerate(ps)
        if moments !== nothing && haskey(moments, p.name)
            m1s[i], m2s[i] = moments[p.name]
        else
            m1s[i] = NeuroDSL.Backend.zeros32(dev, size(p.value)...)
            m2s[i] = NeuroDSL.Backend.zeros32(dev, size(p.value)...)
        end
    end
    train_losses = Float64[]
    val_history = Tuple{Int,Float64}[]
    norm_history = NamedTuple[]   # (; step, norms::Dict{Symbol,Float64}) -- vide si norm_watch vide

    t_start = time()
    for t in (t0+1):(t0+n_steps)
        tokens, labels = sample_window(rng, train_ids, block_size)
        NeuroDSL.set!(g, :token_ids, tokens; atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.set!(g, :labels, labels; atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.invalidate_all!(g; namespace=ns)
        loss_val = NeuroDSL.demand!(g, :loss; namespace=ns)
        push!(train_losses, Float64(sum(Array(loss_val))))
        NeuroDSL.backward_graph!(g, :loss; namespace=ns)
        NeuroDSL.adamw_step_batched!(dev, [p.value for p in ps], [p.gradient for p in ps],
                                     m1s, m2s, lr, b1, b2, eps_v, t, clip, wd)
        NeuroDSL.invalidate_all!(g; namespace=ns)

        if t % val_every == 0 || t == t0 + 1
            vl = val_loss(g, ns; val_ids=val_ids, block_size=block_size)
            push!(val_history, (t, vl))
            @printf("step %6d | train %.4f | val %.4f | ppl %.2f | bits/char %.3f\n",
                    t, train_losses[end], vl, exp(vl), vl/log(2))
            if !isempty(norm_watch)
                norms = Dict{Symbol,Float64}(s => norm(Array(NeuroDSL.node(g, s; namespace=ns).value)) for s in norm_watch)
                push!(norm_history, (; step=t, norms))
            end
        end
        if t in sample_steps
            s = generate_text(g, logits_sym, ns, stoi, chars; block_size=block_size, rng=MersenneTwister(777))
            println("\n--- Échantillon @ pas $t ---\n", s, "\n")
        end
    end
    elapsed = time() - t_start
    final_moments = Dict{Symbol,Tuple{Any,Any}}(p.name => (m1s[i], m2s[i]) for (i, p) in enumerate(ps))

    return (; train_losses, val_history, elapsed, moments=final_moments, rng, norm_history)
end

rng_A = MersenneTwister(123)
n_steps_A = 10_000
result_A = train_char_lm!(g, ns, logits_sym, stoi, chars;
                           train_ids=train_ids, val_ids=val_ids, block_size=block_size,
                           n_steps=n_steps_A, lr=1f-3, rng=rng_A, t0=0)
@printf("\nPhase A terminée : %d pas en %.1f s (%.2f ms/pas moyen)\n",
        n_steps_A, result_A.elapsed, 1000*result_A.elapsed/n_steps_A)

step      1 | train 4.2691 | val 3.7906 | ppl 44.28 | bits/char 5.469
step    500 | train 2.5846 | val 2.5555 | ppl 12.88 | bits/char 3.687
step   1000 | train 2.3979 | val 2.5401 | ppl 12.68 | bits/char 3.665
step   1500 | train 2.4737 | val 2.5249 | ppl 12.49 | bits/char 3.643
step   2000 | train 2.2723 | val 2.3262 | ppl 10.24 | bits/char 3.356
step   2500 | train 2.0915 | val 2.1568 | ppl 8.64 | bits/char 3.112
step   3000 | train 1.9433 | val 2.1173 | ppl 8.31 | bits/char 3.055
step   3500 | train 1.9298 | val 2.0441 | ppl 7.72 | bits/char 2.949
step   4000 | train 1.9479 | val 2.0112 | ppl 7.47 | bits/char 2.902
step   4500 | train 1.7670 | val 1.9471 | ppl 7.01 | bits/char 2.809
step   5000 | train 1.7599 | val 1.8548 | ppl 6.39 | bits/char 2.676
step   5500 | train 1.8247 | val 1.8569 | ppl 6.40 | bits/char 2.679
step   6000 | train 1.5622 | val 1.8154 | ppl 6.14 | bits/char 2.619
step   6500 | train 1.5684 | val 1.8292 | ppl 6.23 | bits/char 2.639
step   7000 | train 1.6225 | 

## 9. Sélection et gel de 6 fenêtres de test (plafond-aware, diverses)

**v2 -- corrige un défaut trouvé après coup dans la v1** : sélectionner par
plus grande taille d'effet seule choisit mécaniquement les cas où le circuit
4-couches est déjà quasi-saturé (recovery 0.95-1.0), laissant zéro marge pour
qu'une couche greffée puisse jamais y contribuer. v2 ajoute un **indice de
plafond** (`ceiling`) : la recovery obtenue en patchant d'un coup les 4 têtes
de couche 1 -- si ce plafond est déjà proche de 1, il ne reste structurellement
rien à expliquer sur cette fenêtre. Sélection : **une seule fenêtre par nom de
personnage** (règle dure -- en v1, les 3 fenêtres avaient toutes atterri sur
"PETRUCHIO"), triées par plafond croissant (les moins saturées en premier), au
nombre de **6** (au lieu de 3). Une **variante "long_gap"** (3 en-têtes A→B→A,
nom différent entre les deux occurrences cibles -- copie à plus longue portée
avec distracteur) est aussi détectée et taguée, potentiellement moins couverte
par des têtes à portée courte.

In [9]:
using LinearAlgebra

# ── Énumération de TOUS les en-têtes de locuteur du split val, position absolue. ──
function all_speaker_headers(val_ids::Vector{Int}, chars::Vector{Char}; min_name_len::Int=4)
    text_val = decode(val_ids, chars)
    headers = NamedTuple[]
    for m in eachmatch(r"\n([A-Z][A-Z ]{2,})\:", text_val)
        name = String(strip(m.captures[1]))
        length(name) >= min_name_len || continue
        push!(headers, (; pos=m.offset + 1, name))
    end
    return headers
end

headers = all_speaker_headers(val_ids, chars)
println("En-têtes de locuteur trouvés dans le split val : ", length(headers))

# ── Candidats : paires de même nom tenant dans une fenêtre de block_size chars.
# "long_gap" si un en-tête de nom DIFFÉRENT s'intercale entre les deux (A→B→A).
function build_candidates(headers, block_size::Int; k::Int=3, margin::Int=5)
    candidates = NamedTuple[]
    n = length(headers)
    for i in 1:n-1
        for jx in i+1:n
            gap = headers[jx].pos - headers[i].pos
            gap > block_size - 20 && break   # headers triés par position -- au-delà, ça ne peut qu'empirer
            headers[i].name != headers[jx].name && continue
            kk = min(k, length(headers[i].name) - 1)
            kk < 1 && continue
            window_start = headers[i].pos - margin
            window_start < 1 && continue
            p1 = headers[i].pos - window_start + 1
            p2 = headers[jx].pos - window_start + 1
            p2 + kk > block_size && continue
            has_intervening_different = any(headers[m2].pos > headers[i].pos && headers[m2].pos < headers[jx].pos &&
                                             headers[m2].name != headers[i].name for m2 in i+1:jx-1)
            variant = has_intervening_different ? :long_gap : :adjacent
            push!(candidates, (; window_start, p1, p2, k=kk, name=headers[i].name, gap, variant))
        end
    end
    return candidates
end

candidates_raw = build_candidates(headers, block_size)
println("Candidats bruts (paires de même nom dans une fenêtre) : ", length(candidates_raw),
        "  (dont long_gap : ", count(c -> c.variant == :long_gap, candidates_raw), ")")
@assert length(candidates_raw) >= 6 "Pas assez de candidats -- revoir le protocole avant de continuer"

# Taille d'effet + indice de PLAFOND (recovery en patchant d'un coup les 4
# têtes de couche 1 -- mesure si le circuit 4-couches sature déjà cette fenêtre),
# tous les deux mesurés sur le modèle DÉJÀ entraîné (Phase A).
function effect_and_ceiling(g, ns, logits_sym, tokens_clean, tokens_corrupt, block_size, j, n_heads::Int)
    NeuroDSL.set!(g, :token_ids, tokens_clean; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    clean_logits = copy(Array(NeuroDSL.demand!(g, logits_sym; namespace=ns)))
    clean_cache  = NeuroDSL.capture_activations(g, ns)

    NeuroDSL.set!(g, :token_ids, tokens_corrupt; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    corrupt_logits = copy(Array(NeuroDSL.demand!(g, logits_sym; namespace=ns)))

    effect = norm(clean_logits[j, :] .- corrupt_logits[j, :])

    layer1_heads = [Symbol(:layer_1_mha_ao_h, h) for h in 1:n_heads]
    NeuroDSL.patch_nodes!(g, layer1_heads, clean_cache; namespace=ns)
    patched_logits = Array(NeuroDSL.demand!(g, logits_sym; namespace=ns))
    ceiling = NeuroDSL.recovery_metric(patched_logits[j:j, :], clean_logits[j:j, :], corrupt_logits[j:j, :])

    NeuroDSL.set!(g, :token_ids, tokens_corrupt; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    NeuroDSL.demand!(g, logits_sym; namespace=ns)
    return effect, ceiling
end

scored = NamedTuple[]
rng_scan = MersenneTwister(999)
for c in candidates_raw
    tokens_clean = val_ids[c.window_start:c.window_start+block_size-1]
    j = c.p2 + c.k - 1
    (j < 1 || j > block_size) && continue
    tokens_corrupt = copy(tokens_clean)
    orig_id = tokens_corrupt[c.p1 + c.k]
    new_id = orig_id
    while new_id == orig_id
        new_id = rand(rng_scan, 1:vocab_size)
    end
    tokens_corrupt[c.p1 + c.k] = new_id
    effect, ceiling = effect_and_ceiling(g, ns, logits_sym, tokens_clean, tokens_corrupt, block_size, j, n_heads)
    push!(scored, (; c..., j, tokens_clean, tokens_corrupt, effect, ceiling))
end
println("Candidats scorés (effet + plafond) : ", length(scored))

effect_max = maximum(x -> x.effect, scored)
effect_floor = max(1.0, 0.25 * effect_max)
eligible = filter(x -> x.effect >= effect_floor, scored)
println("Candidats au-dessus du seuil d'effet (", round(effect_floor, digits=3), ") : ", length(eligible))

# Un seul candidat par nom -- le MOINS saturé (plafond le plus bas) parmi ceux
# au-dessus du seuil d'effet. Corrige directement le défaut "3x le même nom" de v1.
best_per_name = Dict{String,eltype(eligible)}()
for x in eligible
    if !haskey(best_per_name, x.name) || x.ceiling < best_per_name[x.name].ceiling
        best_per_name[x.name] = x
    end
end
by_ceiling = sort(collect(values(best_per_name)), by = x -> x.ceiling)
println("Noms de personnages distincts disponibles : ", length(by_ceiling))

K = min(6, length(by_ceiling))
frozen_windows = by_ceiling[1:K]
n_long_gap = count(x -> x.variant == :long_gap, frozen_windows)

println("\nFenêtres GELÉES (", K, ", triées par plafond croissant -- une seule par nom) :")
for fw in frozen_windows
    @printf("  nom=%-14s variant=%-9s gap=%-4d k=%d  j=%-4d  effet=%.3f  plafond=%.4f\n",
            fw.name, fw.variant, fw.gap, fw.k, fw.j, fw.effect, fw.ceiling)
end
println("Fenêtres 'long_gap' parmi les gelées : ", n_long_gap, "/", K)
if minimum(x -> x.ceiling, frozen_windows) > 0.7
    println("\n⚠️  Aucune fenêtre gelée n'est sous le plafond de 0.7 -- l'induction char-level semble")
    println("   déjà largement saturée par la couche 1 dès ", n_steps_A, " pas. On gèle quand même les")
    println("   6 fenêtres les MOINS saturées disponibles, et on rapporte cette limite honnêtement.")
end

En-têtes de locuteur trouvés dans le split val : 882
Candidats bruts (paires de même nom dans une fenêtre) : 512  (dont long_gap : 482)
Candidats scorés (effet + plafond) : 512
Candidats au-dessus du seuil d'effet (1.0) : 55
Noms de personnages distincts disponibles : 15

Fenêtres GELÉES (6, triées par plafond croissant -- une seule par nom) :
  nom=PETRUCHIO      variant=long_gap  gap=56   k=3  j=64    effet=1.254  plafond=0.5282
  nom=BIONDELLO      variant=long_gap  gap=59   k=3  j=67    effet=1.181  plafond=0.7814
  nom=SEBASTIAN      variant=long_gap  gap=69   k=3  j=77    effet=1.121  plafond=0.7881
  nom=MIRANDA        variant=long_gap  gap=74   k=3  j=82    effet=1.237  plafond=0.8014
  nom=GRUMIO         variant=adjacent  gap=38   k=3  j=46    effet=1.173  plafond=0.8502
  nom=CURTIS         variant=long_gap  gap=31   k=3  j=39    effet=2.789  plafond=0.9147
Fenêtres 'long_gap' parmi les gelées : 5/6


## 10. Recherche de circuit — BASELINE (avant greffe, 4 couches, pas 10 000)

`greedy_patch_search!` + `backward_prune!` (patch de tête ENTIÈRE), avec le
kwarg `metric` restreignant la mesure à la ligne cible `j` de chacune des 6
fenêtres gelées. `max_sites` porté à 8 (au lieu de 6) -- avec 6 fenêtres
potentiellement moins saturées, la recherche peut avoir besoin de plus de
sites pour approcher la recovery maximale.

In [10]:
"""
    find_circuit!(g, ns, logits_sym, window; max_sites=6)

Recherche de circuit sur une fenêtre gelée : capture les caches propre/corrompu
LOCALEMENT (variables locales, jamais de globale réutilisée entre appels --
aucune fuite possible d'un cache périmé après une mutation de graphe entre deux
appels), candidats = toutes les sorties de tête `*_mha_ao_h{h}` du graphe
COURANT (16 avant greffe, 20 après), métrique = recovery restreinte à la ligne
`window.j` (via le nouveau kwarg `metric`, sinon la sortie entière noierait un
effet localisé à une seule position dans une séquence de 256 caractères).
`greedy_patch_search!`/`backward_prune!` réappliquent maintenant `selected` à
chaque mutation (correctif du 2026-07-10) -- sûr même si un site tardif est
retenu avant qu'un site précoce ne soit testé, exactement le cas d'un vrai
circuit d'induction. Remet le graphe en état propre avant de retourner.
"""
function find_circuit!(g::NeuroDSL.NeuroGraph, ns::Symbol, logits_sym::Symbol, window; max_sites::Int=6)
    tokens_clean, tokens_corrupt, j = window.tokens_clean, window.tokens_corrupt, window.j

    NeuroDSL.set!(g, :token_ids, tokens_clean; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    clean_output = copy(Array(NeuroDSL.demand!(g, logits_sym; namespace=ns)))
    clean_cache  = NeuroDSL.capture_activations(g, ns)   # demand! (pas demand_release!) -- capture_activations a besoin des valeurs intermédiaires

    NeuroDSL.set!(g, :token_ids, tokens_corrupt; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    corrupted_output = copy(Array(NeuroDSL.demand!(g, logits_sym; namespace=ns)))
    corrupted_cache  = NeuroDSL.capture_activations(g, ns)

    row_metric(out) = NeuroDSL.recovery_metric(Array(out)[j:j, :], clean_output[j:j, :], corrupted_output[j:j, :])

    candidates = sort(collect(filter(s -> occursin(r"_mha_ao_h\d+$", String(s)), keys(g.nodes[ns]))))

    NeuroDSL.set!(g, :token_ids, tokens_corrupt; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    NeuroDSL.demand!(g, logits_sym; namespace=ns)

    selected, trajectory = NeuroDSL.greedy_patch_search!(g, logits_sym, candidates, clean_cache, corrupted_cache,
                                                           clean_output, corrupted_output;
                                                           namespace=ns, max_sites=max_sites, metric=row_metric)
    remaining, pruned = if isempty(selected)
        (Symbol[], Symbol[])
    else
        NeuroDSL.backward_prune!(g, logits_sym, selected, clean_cache, corrupted_cache,
                                  clean_output, corrupted_output; namespace=ns, metric=row_metric)
    end

    # Vérification indépendante (ancre de sanité) : recovery du sous-ensemble
    # final recalculée depuis un état frais, via patch_nodes! direct.
    NeuroDSL.set!(g, :token_ids, tokens_corrupt; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    NeuroDSL.demand!(g, logits_sym; namespace=ns)
    isempty(remaining) || NeuroDSL.patch_nodes!(g, remaining, clean_cache; namespace=ns)
    out_check = NeuroDSL.demand!(g, logits_sym; namespace=ns)
    r_check = row_metric(out_check)

    # Remise en état propre (texte clean) pour la suite du notebook.
    NeuroDSL.set!(g, :token_ids, tokens_clean; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    NeuroDSL.demand!(g, logits_sym; namespace=ns)

    return (; candidates, selected, trajectory, remaining, pruned, r_check,
            n_candidates=length(candidates), clean_cache, corrupted_cache, clean_output, corrupted_output)
end

baseline_results = [find_circuit!(g, ns, logits_sym, w; max_sites=8) for w in frozen_windows]

println("Recherche de circuit -- BASELINE (4 couches, pas ", n_steps_A, ") :")
for (w, r) in zip(frozen_windows, baseline_results)
    println("  fenêtre '", w.name, "' (", w.variant, ", plafond=", round(w.ceiling,digits=3), ") : sites retenus après élagage = ", r.remaining,
            "  recovery vérifiée = ", round(r.r_check, digits=4), "  (", r.n_candidates, " candidats)")
end

Recherche de circuit -- BASELINE (4 couches, pas 10000) :
  fenêtre 'PETRUCHIO' (long_gap, plafond=0.528) : sites retenus après élagage = [:layer_4_mha_ao_h3, :layer_1_mha_ao_h1, :layer_1_mha_ao_h4, :layer_1_mha_ao_h3, :layer_1_mha_ao_h2, :layer_2_mha_ao_h2, :layer_2_mha_ao_h1]  recovery vérifiée = 0.9949  (16 candidats)
  fenêtre 'BIONDELLO' (long_gap, plafond=0.781) : sites retenus après élagage = [:layer_4_mha_ao_h2, :layer_1_mha_ao_h3, :layer_1_mha_ao_h4, :layer_1_mha_ao_h1, :layer_1_mha_ao_h2, :layer_4_mha_ao_h3, :layer_2_mha_ao_h2, :layer_2_mha_ao_h1]  recovery vérifiée = 0.9967  (16 candidats)
  fenêtre 'SEBASTIAN' (long_gap, plafond=0.788) : sites retenus après élagage = [:layer_4_mha_ao_h1, :layer_1_mha_ao_h2, :layer_4_mha_ao_h2, :layer_1_mha_ao_h3, :layer_1_mha_ao_h1, :layer_3_mha_ao_h1, :layer_4_mha_ao_h3]  recovery vérifiée = 0.9806  (16 candidats)
  fenêtre 'MIRANDA' (long_gap, plafond=0.801) : sites retenus après élagage = [:layer_3_mha_ao_h1, :layer_1_mha_ao_h4, :layer_1

## 11. Chirurgie à chaud — insertion d'une 5ème couche EN COURS d'entraînement

**v2 -- point d'insertion changé** : `:layer_1_out` (au lieu de `:layer_2_out`
en v1). Justifié par les données de la baseline : les circuits mesurés
combinent des têtes de couche 1 (copie) avec des têtes de couches 2-4 (lecture
tardive) -- insérer juste après la couche 1 place la greffe exactement là où
l'information entrerait dans le circuit, en amont de tous les sites de lecture
déjà identifiés. Probabilité d'interception maximale, si une interception est
possible du tout. `insert_block!` insère un `LlamaBlock` (`batched_attn=true`,
homogène), initialisé pour calculer l'IDENTITÉ EXACTE. Preuve F1 inchangée :
texte généré ET val loss comparés exactement avant/après.

In [11]:
n_params_before_graft = length(NeuroDSL.params(g; namespace=ns))

println("--- Échantillon JUSTE AVANT insertion (4 couches) ---")
sample_before_graft = generate_text(g, logits_sym, ns, stoi, chars; block_size=block_size,
                                     rng=MersenneTwister(777), n_chars=200)
val_before_graft = val_loss(g, ns; val_ids=val_ids, block_size=block_size)
println(sample_before_graft)
@printf("val loss juste avant insertion : %.8f\n", val_before_graft)

# ── La chirurgie elle-même : une ligne. Point d'insertion v2 : :layer_1_out. ──
graft_after_sym = :layer_1_out
new_out = NeuroDSL.insert_block!(g, ns, graft_after_sym, dim, n_heads, hidden_dim; batched_attn=true)
graft_prefix = Symbol(:surgery_, graft_after_sym)
println("\nCouche insérée après :", graft_after_sym, " -> nouveau symbole de sortie : ", new_out,
        "  (préfixe : ", graft_prefix, ")")

println("\n--- Échantillon JUSTE APRÈS insertion (5 couches) ---")
sample_after_graft = generate_text(g, logits_sym, ns, stoi, chars; block_size=block_size,
                                    rng=MersenneTwister(777), n_chars=200)
val_after_graft = val_loss(g, ns; val_ids=val_ids, block_size=block_size)
println(sample_after_graft)
@printf("val loss juste après insertion  : %.8f\n", val_after_graft)

# ── Preuve F1 : identité exacte, pas approximative. ──────────────────────
@assert sample_before_graft == sample_after_graft "F1 violée : le texte généré a changé après une insertion censée être l'identité exacte"
@assert val_before_graft == val_after_graft "F1 violée : la val loss a changé après l'insertion"
println("\n✅ F1 confirmée : texte généré ET val loss strictement identiques avant/après insertion.")

n_params_after_graft = length(NeuroDSL.params(g; namespace=ns))
println("Paramètres : ", n_params_before_graft, " -> ", n_params_after_graft,
        "  (+", n_params_after_graft - n_params_before_graft, " nouveaux, apportés par la greffe)")

--- Échantillon JUSTE AVANT insertion (4 couches) ---
BOLTYBALT:
Ready, not I are go what of it hate a thous!
O noble preyer. Gommen! I will go Furth,
With not thine'e appear that and and to't on time beguing
Withou the hoabista'en of way, and your my th
val loss juste avant insertion : 1.68779008

Couche insérée après :layer_1_out -> nouveau symbole de sortie : surgery_layer_1_out_out  (préfixe : surgery_layer_1_out)

--- Échantillon JUSTE APRÈS insertion (5 couches) ---
BOLTYBALT:
Ready, not I are go what of it hate a thous!
O noble preyer. Gommen! I will go Furth,
With not thine'e appear that and and to't on time beguing
Withou the hoabista'en of way, and your my th
val loss juste après insertion  : 1.68779008

✅ F1 confirmée : texte généré ET val loss strictement identiques avant/après insertion.
Paramètres : 40 -> 49  (+9 nouveaux, apportés par la greffe)


## 12. Phase B — 30 000 pas (5 couches)

**v2 -- budget triplé** (10 000 -> 30 000 pas) : le grief de la v1 était que la
couche greffée n'avait eu que 10 000 pas pour rentabiliser sa capacité
supplémentaire. `rng`/`moments` transmis tels quels depuis la Phase A,
`t0=n_steps_A` -- le compteur AdamW global continue sans jamais repasser à 1.
`norm_watch` surveille les 2 poids de sortie de la couche greffée tous les
`val_every` pas -- donne la trajectoire de "sortie de l'identité", pas
seulement la valeur finale.

In [12]:
n_steps_B = 30_000
graft_output_W_sym = Symbol(graft_prefix, :_mha_output_W)
graft_mlp_w2_sym   = Symbol(graft_prefix, :_mlp_w2)

result_B = train_char_lm!(g, ns, logits_sym, stoi, chars;
                           train_ids=train_ids, val_ids=val_ids, block_size=block_size,
                           n_steps=n_steps_B, lr=1f-3, rng=result_A.rng, t0=n_steps_A,
                           moments=result_A.moments,
                           norm_watch=[graft_output_W_sym, graft_mlp_w2_sym])
@printf("\nPhase B terminée : %d pas en %.1f s (%.2f ms/pas moyen)\n",
        n_steps_B, result_B.elapsed, 1000*result_B.elapsed/n_steps_B)

final_val_loss = result_B.val_history[end][2]
@printf("Val loss finale (pas %d, 5 couches) : %.4f nats/char  (ppl %.2f)\n",
        n_steps_A + n_steps_B, final_val_loss, exp(final_val_loss))
println("(Référence 4 couches/20000 pas, real_llm.ipynb : val loss finale 1.6229)")

step  10001 | train 1.4760 | val 1.7124 | ppl 5.54 | bits/char 2.470
step  10500 | train 1.5621 | val 1.7072 | ppl 5.51 | bits/char 2.463
step  11000 | train 1.4494 | val 1.6917 | ppl 5.43 | bits/char 2.441
step  11500 | train 1.6965 | val 1.6794 | ppl 5.36 | bits/char 2.423
step  12000 | train 1.4877 | val 1.6868 | ppl 5.40 | bits/char 2.434
step  12500 | train 1.4059 | val 1.6869 | ppl 5.40 | bits/char 2.434
step  13000 | train 1.5720 | val 1.6743 | ppl 5.33 | bits/char 2.415
step  13500 | train 1.4278 | val 1.6672 | ppl 5.30 | bits/char 2.405
step  14000 | train 1.1604 | val 1.6806 | ppl 5.37 | bits/char 2.425
step  14500 | train 1.5676 | val 1.6437 | ppl 5.17 | bits/char 2.371
step  15000 | train 1.4137 | val 1.6626 | ppl 5.27 | bits/char 2.399
step  15500 | train 1.4194 | val 1.6547 | ppl 5.23 | bits/char 2.387
step  16000 | train 1.4358 | val 1.6251 | ppl 5.08 | bits/char 2.345
step  16500 | train 1.2743 | val 1.6243 | ppl 5.07 | bits/char 2.343
step  17000 | train 1.5347 | val 1

## 13. Recherche de circuit — FINAL (5 couches, pas 40 000) + comparaison

Mêmes 6 fenêtres GELÉES (jamais re-sélectionnées après la greffe), mêmes
corruptions, `find_circuit!` appelé sur le graphe maintenant à 5 couches (20
têtes candidates au lieu de 16). Caches recapturés localement à cet instant
précis -- aucune fuite possible depuis les caches de la baseline, qui
référençaient une topologie de graphe désormais périmée.

In [13]:
final_results = [find_circuit!(g, ns, logits_sym, w; max_sites=8) for w in frozen_windows]

println("\n" * "="^70)
println("COMPARAISON BASELINE (4 couches, pas ", n_steps_A, ") vs FINAL (5 couches, pas ", n_steps_A+n_steps_B, ")")
println("="^70)
for (w, rb, rf) in zip(frozen_windows, baseline_results, final_results)
    println("\nFenêtre '", w.name, "' (", w.variant, ", plafond baseline=", round(w.ceiling, digits=3), ")")
    println("  baseline : sites retenus après élagage = ", rb.remaining,
            "  recovery=", round(rb.r_check, digits=4), "  (", rb.n_candidates, " candidats testés)")
    println("  final    : sites retenus après élagage = ", rf.remaining,
            "  recovery=", round(rf.r_check, digits=4), "  (", rf.n_candidates, " candidats testés)")
    grafted_in_final = filter(s -> occursin("surgery_", String(s)), rf.remaining)
    println("  têtes greffées retenues : ", isempty(grafted_in_final) ? "aucune" : grafted_in_final)
end


COMPARAISON BASELINE (4 couches, pas 10000) vs FINAL (5 couches, pas 40000)

Fenêtre 'PETRUCHIO' (long_gap, plafond baseline=0.528)
  baseline : sites retenus après élagage = [:layer_4_mha_ao_h3, :layer_1_mha_ao_h1, :layer_1_mha_ao_h4, :layer_1_mha_ao_h3, :layer_1_mha_ao_h2, :layer_2_mha_ao_h2, :layer_2_mha_ao_h1]  recovery=0.9949  (16 candidats testés)
  final    : sites retenus après élagage = [:layer_1_mha_ao_h4, :layer_1_mha_ao_h2, :layer_4_mha_ao_h3, :layer_1_mha_ao_h1, :layer_1_mha_ao_h3, :layer_3_mha_ao_h2, :layer_2_mha_ao_h2, :layer_2_mha_ao_h4]  recovery=0.9903  (20 candidats testés)
  têtes greffées retenues : aucune

Fenêtre 'BIONDELLO' (long_gap, plafond baseline=0.781)
  baseline : sites retenus après élagage = [:layer_4_mha_ao_h2, :layer_1_mha_ao_h3, :layer_1_mha_ao_h4, :layer_1_mha_ao_h1, :layer_1_mha_ao_h2, :layer_4_mha_ao_h3, :layer_2_mha_ao_h2, :layer_2_mha_ao_h1]  recovery=0.9967  (16 candidats testés)
  final    : sites retenus après élagage = [:layer_4_mha_ao_h3, 

## 14. Verdict : la couche greffée a-t-elle acquis un rôle causal ?

Critères falsifiables (Fable) : **positif** = normes de sortie greffées non
nulles ET au moins une tête greffée retenue après élagage sur ≥2/6 fenêtres
avec contribution marginale ≥0.05 ET signature d'attention interprétable ;
**négatif** = normes ~0 ou aucune tête greffée jamais retenue, trajectoires
≈ baseline ; **mitigé** = retenue puis élaguée, ou contribution sous le
seuil. Rapporté tel quel, quel que soit le résultat. v2 : sweep individuel
des têtes greffées sur les **6 fenêtres** (pas juste la dernière), et
trajectoire complète des normes pendant la Phase B (`norm_watch`).

In [14]:
norm_output_W = norm(Array(NeuroDSL.node(g, graft_output_W_sym; namespace=ns).value))
norm_mlp_w2   = norm(Array(NeuroDSL.node(g, graft_mlp_w2_sym; namespace=ns).value))
orig_norms_output_W = [norm(Array(NeuroDSL.node(g, Symbol(:layer_,i,:_mha_output_W); namespace=ns).value)) for i in 1:4]

println("‖output_W‖ greffé = ", round(norm_output_W, digits=4),
        "   (moyenne des 4 couches d'origine = ", round(mean(orig_norms_output_W), digits=4), ")")
println("‖mlp_w2‖ greffé   = ", round(norm_mlp_w2, digits=4))

println("\nTrajectoire des normes de sortie greffées pendant la Phase B :")
for nh in result_B.norm_history
    @printf("  pas %6d : ‖output_W‖=%.4f  ‖mlp_w2‖=%.4f\n",
            nh.step, nh.norms[graft_output_W_sym], nh.norms[graft_mlp_w2_sym])
end

# Sweep individuel des 4 têtes greffées SUR LES 6 FENÊTRES (pas juste la
# dernière) -- combien chaque tête greffée récupère-t-elle SEULE ?
grafted_heads = filter(s -> occursin(Regex(string(graft_prefix) * raw"_mha_ao_h\d+$"), String(s)), final_results[1].candidates)
println("\nSweep individuel des têtes greffées (", length(grafted_heads), " têtes × ", length(frozen_windows), " fenêtres) :")
head_recoveries = Dict{Symbol,Dict{Symbol,Float64}}()   # window_name(as Symbol) -> head -> recovery
for (w, rf) in zip(frozen_windows, final_results)
    row_metric_w(out) = NeuroDSL.recovery_metric(Array(out)[w.j:w.j, :], rf.clean_output[w.j:w.j, :], rf.corrupted_output[w.j:w.j, :])
    NeuroDSL.set!(g, :token_ids, w.tokens_corrupt; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    NeuroDSL.demand!(g, logits_sym; namespace=ns)
    wname = Symbol(w.name, "_", w.j)   # clé unique même si un nom revenait (ne devrait plus arriver en v2)
    head_recoveries[wname] = Dict{Symbol,Float64}()
    print("  fenêtre '", w.name, "' : ")
    for h in grafted_heads
        NeuroDSL.patch_node!(g, h, rf.clean_cache; namespace=ns)
        out = NeuroDSL.demand!(g, logits_sym; namespace=ns)
        r = row_metric_w(out)
        head_recoveries[wname][h] = r
        print(h, "=", round(r, digits=4), "  ")
        NeuroDSL.patch_node!(g, h, rf.corrupted_cache; namespace=ns)
        NeuroDSL.demand!(g, logits_sym; namespace=ns)
    end
    println()
    NeuroDSL.set!(g, :token_ids, w.tokens_clean; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    NeuroDSL.demand!(g, logits_sym; namespace=ns)
end

# ── Verdict, contre les critères falsifiables de la cellule précédente ──────
n_windows_with_grafted_retained = count(rf -> !isempty(filter(s -> occursin("surgery_", String(s)), rf.remaining)), final_results)
weights_left_zero = norm_output_W < 1f-3 && norm_mlp_w2 < 1f-3
best_head_recovery = maximum(maximum(values(d); init=0.0) for d in values(head_recoveries); init=0.0)

println("\n" * "="^70)
println("VERDICT")
println("="^70)
println("Poids de sortie greffés restés à zéro (jamais entraînés) ? ", weights_left_zero)
println("Fenêtres où au moins une tête greffée est retenue après élagage : ", n_windows_with_grafted_retained, "/", length(frozen_windows))
println("Meilleure recovery individuelle d'une tête greffée (toutes fenêtres) : ", round(best_head_recovery, digits=4))

verdict = if weights_left_zero
    "NÉGATIF -- la couche greffée n'a jamais quitté l'identité (poids de sortie encore à zéro) : dormante pour cette capacité."
elseif n_windows_with_grafted_retained >= 2 && best_head_recovery >= 0.05
    "POSITIF -- au moins une tête greffée est retenue après élagage sur ≥2 fenêtres, avec une contribution mesurable."
elseif n_windows_with_grafted_retained >= 1 || best_head_recovery >= 0.05
    "MITIGÉ -- une tête greffée a été sélectionnée/montre une contribution individuelle sur au moins une fenêtre, mais sous le seuil de robustesse (≥2 fenêtres, recovery≥0.05)."
else
    "NÉGATIF -- la couche a quitté l'identité (poids non nuls) mais n'a acquis aucun rôle causal détectable dans ce circuit précis, sur aucune des $(length(frozen_windows)) fenêtres."
end
println("\n", verdict)

‖output_W‖ greffé = 28.02   (moyenne des 4 couches d'origine = 32.0073)
‖mlp_w2‖ greffé   = 23.3394

Trajectoire des normes de sortie greffées pendant la Phase B :
  pas  10001 : ‖output_W‖=0.8060  ‖mlp_w2‖=1.1369
  pas  10500 : ‖output_W‖=7.0469  ‖mlp_w2‖=12.9653
  pas  11000 : ‖output_W‖=7.0130  ‖mlp_w2‖=12.6163
  pas  11500 : ‖output_W‖=7.1485  ‖mlp_w2‖=12.4722
  pas  12000 : ‖output_W‖=7.2994  ‖mlp_w2‖=12.3958
  pas  12500 : ‖output_W‖=7.5076  ‖mlp_w2‖=12.4044
  pas  13000 : ‖output_W‖=7.6728  ‖mlp_w2‖=12.3886
  pas  13500 : ‖output_W‖=8.0960  ‖mlp_w2‖=12.5080
  pas  14000 : ‖output_W‖=8.4090  ‖mlp_w2‖=12.6534
  pas  14500 : ‖output_W‖=8.8146  ‖mlp_w2‖=12.8043
  pas  15000 : ‖output_W‖=9.2660  ‖mlp_w2‖=13.0423
  pas  15500 : ‖output_W‖=9.6179  ‖mlp_w2‖=13.2775
  pas  16000 : ‖output_W‖=10.0708  ‖mlp_w2‖=13.6297
  pas  16500 : ‖output_W‖=10.5288  ‖mlp_w2‖=13.8756
  pas  17000 : ‖output_W‖=10.9156  ‖mlp_w2‖=14.0770
  pas  17500 : ‖output_W‖=11.4190  ‖mlp_w2‖=14.2913
  pas  18000 : ‖o

## 15. Run de contrôle : 4 couches à budget total égal (40 000 pas)

La seule référence existante (`real_llm.ipynb`) est à 20 000 pas -- pas
comparable à budget égal avec le run greffé (40 000 pas au total). Un modèle
4 couches frais, entraîné 40 000 pas d'un trait (même graine, même
hyperparamètres), donne le point de comparaison honnête : **la capacité
ajoutée à chaud rapporte-t-elle, à budget de calcul total égal ?**

In [15]:
ns_control = :real_llm_surgery_control
g_control, logits_control = build_char_lm_graph(dev, ns_control; vocab_size=vocab_size, dim=dim, n_heads=n_heads,
                                                  hidden_dim=hidden_dim, n_layers=n_layers, block_size=block_size)

n_steps_control = n_steps_A + n_steps_B   # 40 000 -- budget total identique au run greffé
result_control = train_char_lm!(g_control, ns_control, logits_control, stoi, chars;
                                 train_ids=train_ids, val_ids=val_ids, block_size=block_size,
                                 n_steps=n_steps_control, lr=1f-3, rng=MersenneTwister(123), t0=0)
@printf("\nContrôle 4 couches terminé : %d pas en %.1f s (%.2f ms/pas moyen)\n",
        n_steps_control, result_control.elapsed, 1000*result_control.elapsed/n_steps_control)

control_final_val = result_control.val_history[end][2]
@printf("Val loss finale contrôle (4 couches, %d pas)  : %.4f nats/char\n", n_steps_control, control_final_val)
@printf("Val loss finale greffé    (5 couches, %d pas) : %.4f nats/char\n", n_steps_A+n_steps_B, final_val_loss)
println(final_val_loss < control_final_val ?
        "=> Le run greffé fait MIEUX que le contrôle à budget total égal." :
        "=> Le run greffé ne fait pas mieux que le contrôle à budget total égal (honnête, pas forcé).")

step      1 | train 4.2252 | val 3.7764 | ppl 43.66 | bits/char 5.448
step    500 | train 2.5832 | val 2.5445 | ppl 12.74 | bits/char 3.671
step   1000 | train 2.4076 | val 2.5478 | ppl 12.78 | bits/char 3.676
step   1500 | train 2.5094 | val 2.5181 | ppl 12.41 | bits/char 3.633
step   2000 | train 2.4207 | val 2.4946 | ppl 12.12 | bits/char 3.599
step   2500 | train 2.2426 | val 2.3415 | ppl 10.40 | bits/char 3.378
step   3000 | train 2.0537 | val 2.2282 | ppl 9.28 | bits/char 3.215
step   3500 | train 2.1270 | val 2.1430 | ppl 8.52 | bits/char 3.092
step   4000 | train 2.0536 | val 2.1312 | ppl 8.43 | bits/char 3.075
step   4500 | train 1.8635 | val 2.0287 | ppl 7.60 | bits/char 2.927
step   5000 | train 1.8897 | val 1.9496 | ppl 7.03 | bits/char 2.813
step   5500 | train 1.9280 | val 1.9293 | ppl 6.88 | bits/char 2.783
step   6000 | train 1.6268 | val 1.8973 | ppl 6.67 | bits/char 2.737
step   6500 | train 1.6721 | val 1.9035 | ppl 6.71 | bits/char 2.746
step   7000 | train 1.7038 |

## 16. Courbes combinées (Phase A + Phase B + contrôle, frontière de la greffe marquée)

In [16]:
train_losses_all = vcat(result_A.train_losses, result_B.train_losses)
val_history_all  = vcat(result_A.val_history, result_B.val_history)

plot(1:length(train_losses_all), train_losses_all, label="train loss (greffé, par pas)",
     alpha=0.25, color=:steelblue, xlabel="pas", ylabel="perte (nats/char)",
     title="NeuroDSL -- char-LM sur TinyShakespeare, chirurgie à chaud @ pas $(n_steps_A)",
     legend=:topright, size=(900,500))
plot!(1:length(result_control.train_losses), result_control.train_losses,
      label="train loss (contrôle 4 couches, par pas)", alpha=0.2, color=:seagreen)

val_x = [v[1] for v in val_history_all]
val_y = [v[2] for v in val_history_all]
plot!(val_x, val_y, label="val loss (greffé)", color=:orange, lw=2.5, marker=:circle, markersize=3)

val_x_c = [v[1] for v in result_control.val_history]
val_y_c = [v[2] for v in result_control.val_history]
plot!(val_x_c, val_y_c, label="val loss (contrôle 4 couches)", color=:darkgreen, lw=2, marker=:diamond, markersize=3)

vline!([n_steps_A], label="insertion de la couche", linestyle=:dot, color=:red, lw=2)
hline!([log(vocab_size)], label="ln(vocab_size) -- niveau aléatoire", linestyle=:dash, color=:gray)
savefig(joinpath(@__DIR__, "..", "figures", "real_llm_surgery_v2_loss.pdf"))
println("Figure sauvegardée -> figures/real_llm_surgery_v2_loss.pdf")

Figure sauvegardée -> figures/real_llm_surgery_v2_loss.pdf


## 17. Sauvegarde des résultats

Toutes les métriques clés -- Phase A/B, contrôle, preuve F1, baseline vs
final (6 fenêtres), trajectoire des normes, motifs d'attention, verdict --
dans `real_llm_surgery_v2_results.json`, séparé de la v1 et de `real_llm.ipynb`.

In [17]:
using JSON

# Motif d'attention (pr_h, post-softmax) de la tête la plus forte retenue,
# pour la figure mech-interp classique (clean vs corrompu, le long de j).
function best_head_attention_pattern(g, ns, w, rf)
    isempty(rf.remaining) && return nothing
    site = String(rf.remaining[1])
    pr_sym = Symbol(replace(site, "_ao_h" => "_pr_h"))
    haskey(g.nodes[ns], pr_sym) || return nothing
    NeuroDSL.set!(g, :token_ids, w.tokens_clean; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    NeuroDSL.demand!(g, logits_sym; namespace=ns)
    clean_row = Array(NeuroDSL.node(g, pr_sym; namespace=ns).value)[w.j, :]
    NeuroDSL.set!(g, :token_ids, w.tokens_corrupt; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    NeuroDSL.demand!(g, logits_sym; namespace=ns)
    corrupt_row = Array(NeuroDSL.node(g, pr_sym; namespace=ns).value)[w.j, :]
    NeuroDSL.set!(g, :token_ids, w.tokens_clean; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    NeuroDSL.demand!(g, logits_sym; namespace=ns)
    return (; site, clean_row, corrupt_row)
end
attention_patterns = [best_head_attention_pattern(g, ns, w, rf) for (w, rf) in zip(frozen_windows, final_results)]

surgery_results = Dict(
    "vocab_size" => vocab_size,
    "n_steps_A" => n_steps_A,
    "n_steps_B" => n_steps_B,
    "graft_after_sym" => String(graft_after_sym),
    "elapsed_A_s" => result_A.elapsed,
    "elapsed_B_s" => result_B.elapsed,
    "elapsed_control_s" => result_control.elapsed,
    "train_losses_grafted" => train_losses_all,
    "val_history_grafted" => val_history_all,
    "train_losses_control" => result_control.train_losses,
    "val_history_control" => result_control.val_history,
    "val_loss_mid_training" => result_A.val_history[end][2],
    "val_loss_final" => final_val_loss,
    "val_loss_control_final" => control_final_val,
    "val_loss_reference_4layers_20k" => 1.6229,   # real_llm.ipynb, session précédente
    "f1_identity_confirmed" => (sample_before_graft == sample_after_graft) && (val_before_graft == val_after_graft),
    "n_params_before_graft" => n_params_before_graft,
    "n_params_after_graft" => n_params_after_graft,
    "norm_history" => [(; step=nh.step, output_W=nh.norms[graft_output_W_sym], mlp_w2=nh.norms[graft_mlp_w2_sym])
                        for nh in result_B.norm_history],
    "frozen_windows" => [(; name=w.name, variant=String(w.variant), gap=w.gap, effect=w.effect, ceiling=w.ceiling, j=w.j)
                          for w in frozen_windows],
    "baseline_selected" => [String.(r.remaining) for r in baseline_results],
    "baseline_recovery" => [r.r_check for r in baseline_results],
    "baseline_trajectory" => [[(; site=String(t.site), recovery=t.cumulative_recovery) for t in r.trajectory] for r in baseline_results],
    "final_selected" => [String.(r.remaining) for r in final_results],
    "final_recovery" => [r.r_check for r in final_results],
    "final_trajectory" => [[(; site=String(t.site), recovery=t.cumulative_recovery) for t in r.trajectory] for r in final_results],
    "n_windows_with_grafted_retained" => n_windows_with_grafted_retained,
    "norm_output_W_grafted" => norm_output_W,
    "norm_mlp_w2_grafted" => norm_mlp_w2,
    "norm_output_W_original_mean" => mean(orig_norms_output_W),
    "grafted_head_sweep" => Dict(String(wk) => Dict(String(hk) => hv for (hk,hv) in wv) for (wk,wv) in head_recoveries),
    "attention_patterns" => [ap === nothing ? nothing : (; site=ap.site, clean_row=ap.clean_row, corrupt_row=ap.corrupt_row)
                              for ap in attention_patterns],
    "verdict" => verdict,
)
open(joinpath(@__DIR__, "real_llm_surgery_v2_results.json"), "w") do io
    JSON.print(io, surgery_results)
end
println("Résultats écrits -> notebook/real_llm_surgery_v2_results.json")

Résultats écrits -> notebook/real_llm_surgery_v2_results.json
